# F03 SCRIPTORIUM — PALATIUM
> *Frégate de Rendu — Playwright EGL GPU → Séquence PNG*

**Pipeline :**
1. Monter Drive + valider IN/ (L'Arbitre)
2. Installer Playwright Chromium EGL
3. Lancer le rendu (reprend depuis checkpoint si interruption)
4. Valider OUT_FRAMES/ (L'Arbitre)

---
**Doctrine :** F03 ne lit que `F03_SCRIPTORIUM/IN/` et n'écrit que dans `F03_SCRIPTORIUM/OUT_FRAMES/`.
**Checkpoint sacré :** Toute interruption Colab est récupérable.

In [ ]:
# ─── CONFIGURATION ─────────────────────────────────────────
DRIVE_ROOT  = '/content/drive/MyDrive/DRIVE_PALATIUM'

# Format à rendre : 'shorts' (1080×1920, 30s) ou 'youtube' (3840×2160, 90s)
RENDER_FORMAT = 'shorts'

# Port du serveur Flask statique interne (ne pas changer sauf conflit)
STATIC_PORT = 5003
# ───────────────────────────────────────────────────────────

IN_DIR       = f'{DRIVE_ROOT}/F03_SCRIPTORIUM/IN'
OUT_DIR      = f'{DRIVE_ROOT}/F03_SCRIPTORIUM/OUT_FRAMES'

## Étape 1 — Monter Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive monté')

## Étape 2 — Installer les dépendances

In [ ]:
!pip install flask flask-cors playwright nest_asyncio -q
!playwright install chromium 2>&1 | tail -5
!playwright install-deps chromium 2>&1 | tail -5
print('✅ Playwright + Chromium + nest_asyncio installés')

## Étape 3 — Valider les fichiers IN/ (L'Arbitre CHECK-IN)

In [ ]:
import shutil, subprocess, urllib.request, os

# Téléchargement PAL_ARBITRE.py depuis GitHub
_url = "https://raw.githubusercontent.com/kioka8877-ux/PALATIUM/main/PAL_ARBITRE.py"
urllib.request.urlretrieve(_url, f"{DRIVE_ROOT}/PAL_ARBITRE.py")
shutil.copy(f"{DRIVE_ROOT}/PAL_ARBITRE.py", "/content/PAL_ARBITRE.py")

ret = subprocess.run(["python", "/content/PAL_ARBITRE.py",
    "--frigate", "F03", "--mode", "check-in",
    "--drive-root", DRIVE_ROOT, "--verbose"],
    capture_output=False)

if ret.returncode != 0:
    raise RuntimeError("F03 BLOQUÉE — Transférer les fichiers IN/ avant de continuer")
print("✅ F03 IN/ validé")

## Étape 4 — Vérifier le GPU EGL

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv,noheader
!ls /dev/dri/ 2>/dev/null || echo 'Pas de /dev/dri/ — EGL peut ne pas fonctionner'
print('\nSi vous voyez un GPU T4 ci-dessus, EGL est disponible.')

## Étape 5 — Copier le script de rendu

In [ ]:
import shutil
shutil.copy(f'{DRIVE_ROOT}/F03_SCRIPTORIUM/CODEBASE/pal_f03_render.py', '/content/pal_f03_render.py')
print('✅ pal_f03_render.py copié')

## Étape 6 — Lancer le rendu

> ⚠️ Cette cellule peut durer plusieurs heures selon le format.
> En cas d'interruption Colab, **relancer cette cellule** — le rendu reprend automatiquement depuis le checkpoint.
> Pour recommencer depuis le début, supprimer `OUT_FRAMES/.checkpoint`.

In [ ]:
import sys
sys.path.insert(0, '/content')
from pal_f03_render import run_render

run_render(
    in_dir         = IN_DIR,
    out_frames_dir = OUT_DIR,
    fmt            = RENDER_FORMAT,
    static_port    = STATIC_PORT,
    headless       = True
)

## Étape 7 — Vérifier les frames produites

In [ ]:
from pathlib import Path
from IPython.display import Image, display

frames_dir = Path(OUT_DIR) / RENDER_FORMAT
frames = sorted(frames_dir.glob('frame_*.png'))
print(f'Frames produites : {len(frames)}')

# Afficher quelques frames
preview_indices = [0, len(frames)//4, len(frames)//2, -1] if frames else []
for i in preview_indices:
    if frames:
        print(f'  Frame {frames[i].name}')
        display(Image(str(frames[i]), width=300))

## Étape 8 — Valider OUT_FRAMES/ (L'Arbitre CHECK-OUT)

In [ ]:
import subprocess
ret = subprocess.run(['python', '/content/PAL_ARBITRE.py',
    '--frigate', 'F03', '--mode', 'check-out',
    '--drive-root', DRIVE_ROOT],
    capture_output=False)

if ret.returncode == 0:
    print('\n  ✅ F03 SCRIPTORIUM SCELLÉE — Transférer OUT_FRAMES/ vers F04')
else:
    print('\n  ❌ Vérification échouée — voir messages ci-dessus')

---
## Résumé Transit

Après CHECK-OUT ✅ :

| Source | Destination |
|--------|-------------|
| `F03_SCRIPTORIUM/OUT_FRAMES/` | `F04_EDICTA/IN/OUT_FRAMES/` |

**Inscrire dans `TRACKING/PALATIUM_TRANSFER_LOG.md`.**

*F03 SCRIPTORIUM — Scellée. Ad Victoriam.*